# Anime Data Analysis: A Fan's Perspective

## 1. Introduzione e Definizione dei Task
Questo notebook è stato progettato seguendo i principi dello **User-Centered Design (UCD)**. La nostra *User Persona* di riferimento è il **Fan** dell'animazione giapponese. 

A differenza di un analista di mercato, il Fan esplora i dati spinto da curiosità e passione. I task principali che questa analisi intende supportare sono:
1. Comprendere il rapporto tra popolarità (engagement della community) e qualità percepita (rating).
2. Esplorare le preferenze e i trend all'interno del database.

Seguendo il **Mantra di Shneiderman** (*"Overview first, zoom and filter, then details-on-demand"*), il processo inizierà con un'analisi esplorativa (EDA) per valutare la "salute" dei dati, per poi procedere con visualizzazioni mirate.

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configurazione estetica (Principio HCI: Leggibilità e contrasto)
sns.set_theme(style="whitegrid")

# 1. Caricamento dei dataset dalla cartella dedicata
print("Caricamento dati in corso...")
df_details = pd.read_csv('datasets/details.csv', low_memory=False)
df_stats = pd.read_csv('datasets/stats.csv', low_memory=False)
df_favs = pd.read_csv('datasets/favs.csv', low_memory=False)
print("Caricamento completato!\n")

# 2. Esplorazione Strutturale (Mantra di Shneiderman: Overview first)
def profile_dataframe(df, name):
    print(f"--- Profilazione Dataset: {name} ---")
    print(f"Dimensioni: {df.shape[0]} righe, {df.shape[1]} colonne")
    null_counts = df.isnull().sum()
    print("Valori Nulli (Top 5):")
    print(null_counts[null_counts > 0].sort_values(ascending=False).head(5))
    print("-" * 40 + "\n")

profile_dataframe(df_details, "Details")
profile_dataframe(df_stats, "Stats")
profile_dataframe(df_favs, "Favs")

# 3. Analisi di Sanità dei Dati (Controllo Outliers) - VERSIONE ULTRA-OTTIMIZZATA

# OPTIMIZATION: Evitiamo merge non necessari. Le colonne 'title' e 'scored_by' 
# sono già in df_details. Gestiamo i valori Nulli riempiendoli con 0, 
# poiché rappresentano anime senza alcun voto.
df_details['scored_by'] = df_details['scored_by'].fillna(0)

# Mostriamo i Top 5 anime per numero di voti per individuare i "giganti" (Outliers)
print("--- Top 5 Anime per numero di voti (Sanity Check Ottimizzato) ---")
top_5_anime = df_details[['title', 'scored_by']].sort_values(by='scored_by', ascending=False).head(5)
print(top_5_anime)

Caricamento dati in corso...
Caricamento completato!

--- Profilazione Dataset: Details ---
Dimensioni: 28955 righe, 29 colonne
Valori Nulli (Top 5):
year         22689
season       22689
end_date     17688
scored_by    10073
score        10073
dtype: int64
----------------------------------------

--- Profilazione Dataset: Stats ---
Dimensioni: 28955 righe, 27 colonne
Valori Nulli (Top 5):
score_1_votes         430
score_1_percentage    430
score_2_votes         430
score_2_percentage    430
score_3_votes         430
dtype: int64
----------------------------------------

--- Profilazione Dataset: Favs ---
Dimensioni: 4178747 righe, 3 colonne
Valori Nulli (Top 5):
username    4
dtype: int64
----------------------------------------

--- Top 5 Anime per numero di voti (Sanity Check Ottimizzato) ---
                                  title  scored_by
22010                Shingeki no Kyojin  2979733.0
4788                         Death Note  2919353.0
17990                     One Punch Man

## 2. Exploratory Data Analysis (EDA) e Giustificazione delle Scelte di Design

L'analisi strutturale iniziale sui dataset `details`, `stats` e `favs` ha rivelato importanti asimmetrie nel dominio dei dati, che richiedono specifiche strategie di *Data Cleaning* e *Visualisation* per evitare di presentare informazioni fuorvianti:

* **Gestione dei Missing Data (Bias Prevention):** Su quasi 29.000 titoli, oltre 22.000 mancano dei campi `year` e `season`. Rimuovere queste righe (*Listwise Deletion*) introdurrebbe un grave **Selection Bias**. Questi valori nulli rappresentano una "realtà" del dominio (OVA, Film, Speciali musicali che non seguono le canoniche stagioni televisive). Verranno filtrati solo in caso di analisi strettamente cronologiche.
* **Presenza di Outlier ("Giganti"):** La distribuzione dei voti è fortemente sbilanciata. Titoli come *Death Note* o *One Punch Man* superano i 2 milioni di voti, mentre la maggioranza degli anime ha volumi di interazione ordini di grandezza inferiori.
* **Scelta Architetturale (Log Scale):** Come definito dai principi di *Large Scale Data Visualisation*, mappare questi dati su un grafico lineare genererebbe una rappresentazione "Bad", schiacciando il 99% dei titoli a causa degli outlier. Per la prossima visualizzazione, applicheremo una **Scala Logaritmica** sull'asse delle ascisse per garantire leggibilità e preservare le proporzioni su tutta la scala dei dati.

In [4]:
import plotly.express as px

# Rimuoviamo gli anime che non hanno ricevuto voti o punteggi per evitare errori nel logaritmo
df_valid_scores = df_details[(df_details['score'].notnull()) & (df_details['scored_by'] > 0)].copy()

# Scatter plot Interattivo con Plotly Express
fig = px.scatter(
    df_valid_scores, 
    x='scored_by', 
    y='score', 
    hover_name='title',          # MAGIA: Il titolo dell'anime apparirà in grande nel tooltip!
    hover_data={
        'scored_by': True,       # Mostra i votanti
        'score': True,           # Mostra lo score
        'type': True             # Mostra anche se è TV, Movie, OVA, ecc.
    },
    log_x=True,                  # Applica automaticamente la scala logaritmica sull'asse X
    opacity=0.3,                 # Trasparenza per gestire l'overplotting
    color_discrete_sequence=['#2C3E50'], # Colore dei punti
    title='Rapporto tra Popolarità e Qualità Percepita (Interattivo)'
)

# Miglioriamo l'estetica e la leggibilità (HCI)
fig.update_layout(
    xaxis_title='Numero di Utenti che hanno votato (Log Scale)',
    yaxis_title='Punteggio Medio (Score 1-10)',
    template='plotly_white',     # Sfondo pulito come in Seaborn
    width=900,
    height=600
)

# Aggiungiamo le linee per le mediane, come avevamo fatto prima
mediana_score = df_valid_scores['score'].median()
mediana_popolarita = df_valid_scores['scored_by'].median()

fig.add_hline(y=mediana_score, line_dash="dash", line_color="red", 
              annotation_text=f"Mediana Score: {mediana_score}", annotation_position="top right")
fig.add_vline(x=mediana_popolarita, line_dash="dash", line_color="orange", 
              annotation_text=f"Mediana Votanti: {mediana_popolarita:.0f}", annotation_position="top right")

# Mostriamo il grafico interattivo
fig.show()

## 3. Overview e Distribuzione Globale (Il "Colpo d'occhio")

Per un Fan, il primo interesse è spesso capire come si posizionano i propri anime preferiti rispetto alla media generale. Iniziamo analizzando la distribuzione dei punteggi (`score`). 

* **Scelta di Design (HCI):** Utilizziamo un istogramma con una curva di densità (KDE). Questo permette all'utente di identificare immediatamente dov'è la "maggioranza" dei voti, fornendo un ancoraggio cognitivo prima di esplorare dati più specifici.

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

# Rimuoviamo i valori nulli 
df_voti_validi = df_details[df_details['score'].notnull()].copy()
media_score = df_voti_validi['score'].mean()

# Creiamo l'istogramma
fig = px.histogram(
    df_voti_validi, 
    x='score', 
    color_discrete_sequence=['#FF8C69'] # Colore leggermente più chiaro per simulare Seaborn
)

# 1. CORREZIONE COLONNE E TRASPARENZA
fig.update_traces(
    xbins=dict(start=1, end=10, size=0.2), # Forza le colonne ad essere sottili come in Seaborn
    marker_line_color='black', 
    marker_line_width=1, 
    opacity=0.6, # Aggiunge la trasparenza esatta di "Prima.jpeg"
    hovertemplate='<b>Punteggio:</b> %{x}<br><b>Numero di Anime:</b> %{y}<extra></extra>' 
)

# 2. CORREZIONE LAYOUT E GRIGLIA
fig.update_layout(
    title='<b>Distribuzione dei Punteggi (Score) degli Anime</b>',
    title_x=0.5, # Centra il titolo come in Seaborn
    xaxis_title='Punteggio Medio (Score)',
    yaxis_title='Numero di Anime',
    plot_bgcolor='white', # Sfondo bianco
    width=900, 
    height=540,
    hoverlabel=dict(bgcolor="white", font_size=13, font_family="Arial")
)

# Creiamo il "riquadro" grigio chiaro attorno al grafico (Bounding Box) e la griglia interna
fig.update_xaxes(
    showgrid=True, gridwidth=1, gridcolor='#EBEBEB', 
    showline=True, linewidth=1, linecolor='#CCCCCC', mirror=True # 'mirror=True' chiude il riquadro
)
fig.update_yaxes(
    showgrid=True, gridwidth=1, gridcolor='#EBEBEB', 
    showline=True, linewidth=1, linecolor='#CCCCCC', mirror=True
)

# 3. CORREZIONE DELLA LINEA MEDIA E DELLA LEGENDA
# Aggiungiamo la linea tratteggiata
fig.add_vline(
    x=media_score, 
    line_dash="dash", 
    line_color="blue", 
    line_width=2
)

# Trucco elegante: Aggiungiamo una "finta" traccia invisibile solo per far apparire 
# il riquadro della legenda in alto a destra, identico a quello di Seaborn!
fig.add_trace(go.Scatter(
    x=[None], y=[None],
    mode='lines',
    line=dict(color='blue', width=2, dash='dash'),
    name=f'Media: {media_score:.2f}'
))

# Mostriamo il grafico interattivo
fig.show()

## 4. Esplorazione Interattiva per Genere (Zoom and Filter)

Un task fondamentale per il Fan è cercare nuovi titoli in base ai propri gusti. Il dataset presenta i generi raggruppati in stringhe (es. `['Action', 'Sci-Fi']`). Per permettere il filtraggio, applicheremo una trasformazione dei dati (`explode`) e utilizzeremo `ipywidgets` per creare un menu a tendina interattivo.

* **Scelta di Design (HCI):** Invece di mostrare enormi e confusionarie tabelle con tutti i generi, l'utente può selezionare attivamente il suo genere preferito tramite un **Dropdown**. Il sistema risponderà generando un *Bar Chart* dei 10 anime più popolari per quel genere, riducendo drasticamente il rumore visivo.

In [10]:
import ast
import ipywidgets as widgets
from IPython.display import display
import plotly.express as px

# 1. Data Cleaning: Convertiamo le stringhe dei generi in vere liste Python
df_details['genres_clean'] = df_details['genres'].fillna('[]')
df_details['genres_list'] = df_details['genres_clean'].apply(ast.literal_eval)

# 2. Esplodiamo le liste: un anime con 3 generi diventerà 3 righe separate
df_exploded = df_details.explode('genres_list')

# Otteniamo la lista unica dei generi (rimuovendo i valori nulli/vuoti)
generi_unici = sorted([g for g in df_exploded['genres_list'].unique() if pd.notna(g)])

# 3. Funzione di visualizzazione dinamica interattiva
def mostra_top_anime_per_genere(genere):
    # Filtriamo per genere e ordiniamo per popolarità (scored_by)
    df_genere = df_exploded[df_exploded['genres_list'] == genere]
    top_10 = df_genere.sort_values(by='scored_by', ascending=False).head(10)
    
    # TRUCCO PLOTLY: Plotly disegna i grafici a barre orizzontali partendo dal basso.
    # Invertiamo l'ordine del dataframe per assicurarci che il #1 sia in alto!
    top_10 = top_10.iloc[::-1]
    
    # Creiamo il Bar Chart interattivo
    fig = px.bar(
        top_10, 
        x='scored_by', 
        y='title', 
        orientation='h', # Imposta le barre in orizzontale
        color='scored_by', # Colora in base ai voti
        color_continuous_scale='viridis_r', # Mantiene la tua palette originale
        title=f'<b>Top 10 Anime più popolari del genere: {genere}</b>'
    )
    
    # Personalizziamo il layout e l'estetica
    fig.update_layout(
        xaxis_title='Numero di Voti (Popularity)',
        yaxis_title='Titolo Anime',
        template='plotly_white',
        width=900,
        height=500,
        coloraxis_showscale=False, # Nasconde la barra dei colori laterale per mantenere pulizia
        hoverlabel=dict(bgcolor="white", font_size=14, font_family="Arial")
    )
    
    # Aggiungiamo l'Hover Interattivo (Il dettaglio esatto dei voti)
    # %{x:,} inserisce le virgole per le migliaia (es. 2,145,000) riducendo il carico cognitivo
    fig.update_traces(
        hovertemplate='<b>%{y}</b><br>Voti Totali: <b>%{x:,}</b><extra></extra>',
        marker_line_color='black', # Bordino nero alle barre
        marker_line_width=1
    )
    
    fig.show()

# 4. Creazione dell'Interfaccia (Widget)
dropdown_genere = widgets.Dropdown(
    options=generi_unici,
    value='Action', # Valore di default
    description='Scegli Genere:',
    disabled=False,
)

# Colleghiamo il menu a tendina alla funzione grafica
widgets.interact(mostra_top_anime_per_genere, genere=dropdown_genere);

interactive(children=(Dropdown(description='Scegli Genere:', options=('Action', 'Adventure', 'Avant Garde', 'A…